# 🔬 Forensics Lab — Enveda CASMI 2026

> *Welcome to the Forensics Lab. 400 molecules — 400 cold cases.*
> *You have fingerprints, but zero suspects. Can you identify them all?*

---

## 🕵️ The Case

Modern mass spectrometry can detect **thousands of molecules** in nature — but identifying them is like solving a crime with only a **partial fingerprint**.

Every molecule leaves a **mass spectrum** — a unique fragmentation pattern. Your job: match that "fingerprint" to a suspect in the database... or **draw the suspect from scratch** if they've never been seen before.

---

## 🧩 Mapping the Investigation

| Chemistry | Forensics Analogy |
|---|---|
| MS/MS spectrum | 🖐️ Fingerprint at the crime scene |
| Fragment peaks (m/z) | Papillary ridges of the print |
| Peak intensities | How clearly each ridge imprinted |
| Precursor m/z | Blood type + approximate height |
| Adduct (`[M+H]+`, `[M+Na]+`) | Type of weapon used |
| Collision energy | How smudged the print is |
| Molecular formula | Suspect's build & body type |
| **SMILES structure** | 🎨 The full **wanted poster** |
| InChIKey14 | Identical twin detection — face without scars |
| **Top-25 predictions** | Lineup of 25 suspects |
| **MRR@25** | How high the real culprit stands in the lineup |

---

## 🚨 Three Levels of Difficulty

| Class | Forensics Analogy | Method |
|---|---|---|
| 🟢 **Class 1** | Suspect **already in the police database** | Spectral library search |
| 🟡 **Class 2** | Suspect in **Interpol files**, no mugshot yet | Database retrieval |
| 🔴 **Class 3** | **Ghost** — never registered anywhere | De novo reconstruction |

---

## ⚖️ The Rules of the Game

- 🎯 **400 molecules**, each with 1–16 spectra (median 3)
- 📇 **One suspect per `molecule_id`** — merge all evidence
- 🏆 **MRR@25** scoring — a correct hit at rank 1 = 1.0, at rank 25 = 0.04
- 🎭 **Stereochemistry & tautomers ignored** — only the 2D skeleton counts (InChIKey14)
- ⏱️ **9-hour runtime limit**, no internet, notebook submission

---

## 🗺️ Roadmap — From Cold Case to Closed Case

| Phase | Goal | What We Do |
|---|---|---|
| 🟢 **Day 1** | Launch baseline | Cosine similarity retrieval → `submission.csv` |
| 🟡 **Week 1** | Spectral encoder | Transformer over peak lists + contrastive pretraining |
| 🟠 **Week 2** | Molecular encoder | GNN / MolFormer for structure embeddings |
| 🔴 **Week 3** | De novo head | SMILES decoder for Class 3 ghosts |
| 👑 **Final** | Full pipeline | Retrieval + reranking + ensemble |

---

## 💡 Key Takeaway

> **65.5% of the challenge is retrieval, 35% is true detective work.**
> Class 1 and 2 are solved by *knowing where to look*.
> Class 3 is solved by *knowing how molecules think*.

**Let's open the case file.** 🔍

Hello everyone, my decision is based on two public works that I have uploaded to deepsikand I am sitting here sorting!)

https://www.kaggle.com/code/haideptry/enveda-casmi-2026-fast-spectral-cosine-baseline/notebook


https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline


## About the works this solution is based on

This solution is built on top of two public works, links above.

**1. Fast Spectral Cosine Baseline (haideptry)**

A basic retrieval pipeline: spectral library search via cosine similarity, mass-window filtering, top-25 candidates per molecule. It provides the foundation for Class-1 molecules — those with a public reference spectrum.

**2. Analog Propagation (prvsiyan)**

The key idea is **mass-shifted analog propagation**. For molecules without a reference spectrum (Class 2), structural relatives are found in the training library, and spectra are compared with a mass shift. It is extended with **MetFrag-lite** (in-silico bond fragmentation of the candidate) and a **HistGradientBoosting ranker** on 25 features. This is what lifts the score from 0.15 to 0.266.

Both works are published under Apache 2.0. Full credit to their authors.

## FUTURE PLAN

**Completed**
- Launched a copy of the public work **Analog Propagation (prvsiyan, LB 0.266)**
- 712k candidate pool (COCONUT ∪ train) with precomputed fingerprints
- Mass-shifted analog propagation (±200 Da)
- MetFrag-lite in-silico fragmentation
- HistGradientBoosting ranker on 25 features

**Next Steps**
- Take from **Multi-Spectrum Consensus (LB 0.239)** what is missing in 0.266:
  - **Dual-channel similarity** (fragments + neutral losses, 70/30) instead of a single channel
  - **Multi-spectrum consensus**: `best_blend + 0.12×second + 0.01×support + 0.025×NP_prior` instead of max aggregation
  - **Dynamic Sylva gating**: threshold 0.60 / 0.55 at ≥2 spectra for confident Class-1 filtering
  - **Chemical heuristics** (DBE, O/C, N/C) as additional features for COCONUT candidates
- Build a hybrid: ranker from 0.266 + consensus and dual-channel from 0.239
- Target: **0.28+**

## 1. CONFIG

Все гиперпараметры в одном классе `CFG`: окно массы для кандидатов, очистка спектра, параметры analog propagation, настройки ранкера и число кандидатов в сабмите.

In [1]:
# ===================================================================================
#  CONFIG — every knob in one place. Each is annotated with the measurement behind it.
# ===================================================================================
class CFG:
    # --- candidate generation -------------------------------------------------------
    PPM_WIN      = 10.0   # neutral-mass window for candidates. TIGHTER IS BETTER:
                          # +-10ppm -> 0.521 | +-20 -> 0.509 | +-30 -> 0.500 (Class-2 MRR)
                          # timsTOF precursor error stays under ~9 ppm (+1.4 ppm offset).
    PPM_FALLBACK = 30.0   # only used if the tight window returns nothing at all.

    # --- spectrum cleaning ----------------------------------------------------------
    INT_FLOOR    = 0.002  # drop peaks below this fraction of the base peak
    MAX_PEAKS    = 256    # keep the N most intense peaks after the floor
    MZ_TOL       = 0.01   # Da tolerance when matching two peaks
    INT_POWER    = 1.0    # intensity transform before similarity (1.0 + entropy weighting
                          # beat sqrt: Class-1 0.919 vs 0.895)
    ENT_WEIGHT   = True   # Li et al. 2021 entropy weighting of low-entropy spectra

    # --- analog propagation (the main idea) -----------------------------------------
    ANALOG_WIN   = 200.0  # +- Da mass-shift window. +-400 gave no gain (0.520 vs 0.521).
    N_ANALOG     = 80     # analogs kept per molecule; saturates here (40 -> 0.515)
    SIM_POWER    = 3.0    # sim^p weighting. p=1 -> 0.498, p=3 -> 0.521, p=4 -> 0.525

    # --- ranker ---------------------------------------------------------------------
    W1           = 0.42   # weight on the Class-1 simulation. NOT the class share (0.16) --
                          # see the ranker section: it is the leaderboard-calibrated value.
    GBM = dict(max_depth=4, max_iter=220, learning_rate=0.07,
               min_samples_leaf=60, l2_regularization=1.0)

    TOPN         = 25     # the metric allows 25 guesses; there is no penalty for using them all

## 2. IMPORTS & PATHS

Импорт библиотек, автопоиск путей к train.parquet, test.parquet, sample_submission.csv.

In [2]:
import os, glob, time, pickle, math
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa
T0 = time.time()

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits: raise FileNotFoundError(name)
    return sorted(hits, key=len)[0]

COMP   = os.path.dirname(find('test.parquet'))
TRAIN  = os.path.join(COMP, 'train.parquet')
TEST   = os.path.join(COMP, 'test.parquet')
SAMPLE = os.path.join(COMP, 'sample_submission.csv')
print('competition files:', os.listdir(COMP))

competition files: ['sample_submission.csv', 'train.parquet', 'test.parquet']


## 3. RDKIT INSTALL

Установка офлайн RDKit wheel из приаттаченного датасета. Нужен только для канала in-silico fragmentation, без него ноутбук работает.

In [3]:
# RDKit is not in the Kaggle image and internet is off for code competitions, so install the
# wheel from an attached dataset. Only the fragmentation channel needs it.
import subprocess, sys, glob
whl = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
if whl:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', whl[0]], check=False)
try:
    from rdkit import Chem
    HAVE_RDKIT = True
except Exception:
    HAVE_RDKIT = False
print('RDKit available:', HAVE_RDKIT, '(fragmentation channel is optional - the notebook runs without it)')

RDKit available: True (fragmentation channel is optional - the notebook runs without it)


## 4. SIMILARITY KERNELS

Numba JIT-функции для очистки спектра и скоринга: `_clean` (фильтр пиков + энтропийное взвешивание), `entropy_sim` (Li et al. 2021), `cos_sim`, `search` (параллельный прогон по кандидатам), `entropy_sim_shift` + `search_shift` (mass-shifted матчинг для analog propagation).

In [4]:
"""Similarity kernels: weighted cosine + spectral entropy similarity (Li et al. 2021)."""
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n=len(mz)
    if n==0: return np.empty(0,np.float32), np.empty(0,np.float32)
    mx=0.0
    for i in range(n):
        if it[i]>mx: mx=it[i]
    if mx<=0: return np.empty(0,np.float32), np.empty(0,np.float32)
    thr=floor*mx; c=0
    for i in range(n):
        if it[i]>=thr: c+=1
    idx=np.empty(c,np.int64); j=0
    for i in range(n):
        if it[i]>=thr: idx[j]=i; j+=1
    if c>topk:
        v=np.empty(c,np.float32)
        for i in range(c): v[i]=it[idx[i]]
        o=np.argsort(v)[c-topk:]
        k2=np.empty(topk,np.int64)
        for i in range(topk): k2[i]=idx[o[i]]
        k2.sort(); idx=k2; c=topk
    om=np.empty(c,np.float32); oi=np.empty(c,np.float32)
    s=0.0
    for i in range(c):
        om[i]=mz[idx[i]]; v=it[idx[i]]**power; oi[i]=v; s+=v
    if s>0:
        for i in range(c): oi[i]/=s
    if ent_weight:
        S=0.0
        for i in range(c):
            if oi[i]>0: S-=oi[i]*np.log(oi[i])
        if S<3.0:
            w=0.25+0.25*S; s2=0.0
            for i in range(c): oi[i]=oi[i]**w; s2+=oi[i]
            if s2>0:
                for i in range(c): oi[i]/=s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz,qp,cmz,cp,tol):
    i=0;j=0;n=len(qmz);m=len(cmz)
    SA=0.0
    for x in range(n):
        if qp[x]>0: SA-=qp[x]*np.log(qp[x])
    SB=0.0
    for x in range(m):
        if cp[x]>0: SB-=cp[x]*np.log(cp[x])
    SAB=0.0; tot=0.0
    buf=np.empty(n+m,np.float64); b=0
    while i<n and j<m:
        d=qmz[i]-cmz[j]
        if d<-tol: buf[b]=qp[i]; i+=1; b+=1
        elif d>tol: buf[b]=cp[j]; j+=1; b+=1
        else: buf[b]=qp[i]+cp[j]; i+=1; j+=1; b+=1
    while i<n: buf[b]=qp[i]; i+=1; b+=1
    while j<m: buf[b]=cp[j]; j+=1; b+=1
    for x in range(b): tot+=buf[x]
    if tot<=0: return 0.0
    for x in range(b):
        v=buf[x]/tot
        if v>0: SAB-=v*np.log(v)
    return 1.0-(2.0*SAB-SA-SB)/np.log(4.0)

@njit(cache=True, fastmath=True)
def cos_sim(qmz,qp,cmz,cp,tol):
    i=0;j=0;n=len(qmz);m=len(cmz); dot=0.0; na=0.0; nb=0.0
    for x in range(n): na+=qp[x]*qp[x]
    for x in range(m): nb+=cp[x]*cp[x]
    while i<n and j<m:
        d=qmz[i]-cmz[j]
        if d<-tol: i+=1
        elif d>tol: j+=1
        else: dot+=qp[i]*cp[j]; i+=1; j+=1
    if na<=0 or nb<=0: return 0.0
    return dot/np.sqrt(na*nb)

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz,qp,cand,off,allmz,allin,tol,floor,topk,power,ent_weight,kind):
    out=np.zeros(len(cand),np.float32)
    for k in prange(len(cand)):
        c=cand[k]; a=off[c]; b=off[c+1]
        if b<=a: continue
        cm,cp=_clean(allmz[a:b],allin[a:b],floor,topk,power,ent_weight)
        if len(cm)==0: continue
        out[k]= entropy_sim(qmz,qp,cm,cp,tol) if kind==1 else cos_sim(qmz,qp,cm,cp,tol)
    return out

def prep(mz,it,floor=0.002,topk=256,power=0.5,ent_weight=False):
    return _clean(np.asarray(mz,np.float32),np.asarray(it,np.float32),floor,topk,power,ent_weight)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz,qp,cmz,cp,tol,shift):
    """Best of direct and mass-shifted entropy similarity."""
    a = entropy_sim(qmz,qp,cmz,cp,tol)
    if shift > -0.001 and shift < 0.001: return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)): sm[i]=cmz[i]+shift
    b = entropy_sim(qmz,qp,sm,cp,tol)
    return a if a>b else b

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz,qp,cand,off,allmz,allin,tol,floor,topk,power,ent_weight,kind,shift):
    out=np.zeros(len(cand),np.float32)
    for k in prange(len(cand)):
        c=cand[k]; a=off[c]; b=off[c+1]
        if b<=a: continue
        cm,cp=_clean(allmz[a:b],allin[a:b],floor,topk,power,ent_weight)
        if len(cm)==0: continue
        out[k]=entropy_sim_shift(qmz,qp,cm,cp,tol,shift[k])
    return out

## 5. ADDUCTS, LIBRARY & EVIDENCE CHANNELS

Таблица аддуктов и `neutral_mass` для перевода precursor m/z в нейтральную массу. `load_library` читает train.parquet через PyArrow zero-copy. `build_rep` берёт по одному самому богатому спектру на структуру для analog matching. `lib_sim` — прямой spectral match (Class 1), `analog_sim` — mass-shifted match по окну ±200 Да (Class 2).

In [5]:
MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
            S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
            Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {
 "[M+H]+":(1,1,PROTON), "[M+NH4]+":(1,1,NH4-E), "[M+Na]+":(1,1,MASS['Na']-E),
 "[M+K]+":(1,1,MASS['K']-E), "[M-H2O+H]+":(1,1,PROTON-H2O), "[M-2H2O+H]+":(1,1,PROTON-2*H2O),
 "[M+2H]2+":(1,2,2*PROTON), "[M]+":(1,1,-E), "[M-H2O]+":(1,1,-E-H2O),
 "[M+CH3OH+H]+":(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 "[M+CH3CN+H]+":(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 "[M-H]-":(1,1,-PROTON), "[M-H2O-H]-":(1,1,-PROTON-H2O), "[M+CH2O2-H]-":(1,1,FORMATE-PROTON),
 "[M+C2H4O2-H]-":(1,1,ACETATE-PROTON), "[M+Cl]-":(1,1,MASS['Cl']+E), "[M]-":(1,1,E),
 "[M-2H]-":(1,2,-2*PROTON), "[M+Na-2H]-":(1,1,MASS['Na']-2*PROTON),
 "[2M+H]+":(2,1,PROTON), "[2M+Na]+":(2,1,MASS['Na']-E), "[2M+NH4]+":(2,1,NH4-E),
 "[2M+K]+":(2,1,MASS['K']-E), "[2M-H]-":(2,1,-PROTON), "[2M+CH2O2-H]-":(2,1,FORMATE-PROTON),
 "[2M+C2H4O2-H]-":(2,1,ACETATE-PROTON), "[2M+Na-2H]-":(2,1,MASS['Na']-2*PROTON),
 "[3M+H]+":(3,1,PROTON), "[3M-H]-":(3,1,-PROTON),
}
def neutral_mass(mz, adduct):
    out=np.full(len(mz), np.nan); ad=np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m=(ad==a)
        if m.any(): out[m]=(mz[m]*z-d)/n
    return out


# ===================================================================================
#  Library + candidate pool
# ===================================================================================
def load_library(path):
    t0 = time.time()
    t = pq.read_table(path, columns=['inchikey14','normalized_smiles','adduct','precursor_mz',
                                     'ms2_mzs','ms2_normalized_intensities'])
    mzc = t.column('ms2_mzs').combine_chunks(); itc = t.column('ms2_normalized_intensities').combine_chunks()
    off = mzc.offsets.to_numpy().astype(np.int64)
    allmz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    allin = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    prec = t.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
    add = np.asarray(t.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
    ik  = np.asarray(t.column('inchikey14').cast(pa.string()).to_pylist(), dtype=object)
    smi = np.asarray(t.column('normalized_smiles').cast(pa.string()).to_pylist(), dtype=object)
    nm  = neutral_mass(prec, add); ok = np.isfinite(nm)
    order = np.argsort(np.where(ok, nm, 1e18), kind='mergesort')
    best = {}
    for k, s in zip(ik, smi):
        if k and s and k not in best: best[k] = s
    print(f'library: {len(off)-1:,} spectra / {len(best):,} structures  ({time.time()-t0:.0f}s)', flush=True)
    return dict(off=off, mz=allmz, it=allin, nm=nm, ik=ik, best=best,
                order=order, snm=nm[order], n_ok=int(ok.sum()))

def lib_window(L, target, tol):
    lo = np.searchsorted(L['snm'][:L['n_ok']], target-tol, 'left')
    hi = np.searchsorted(L['snm'][:L['n_ok']], target+tol, 'right')
    return L['order'][lo:hi]

def build_rep(L):
    """One representative spectrum per structure (the richest), sorted by neutral mass.
       Using 3 per structure was WORSE (0.49 vs 0.52): extra spectra raise the max similarity
       of irrelevant structures too, which flattens the discrimination."""
    npk = np.diff(L['off']); best = {}; ik = L['ik']
    for i in range(len(ik)):
        k = ik[i]
        if k and (k not in best or npk[i] > npk[best[k]]): best[k] = i
    rep = np.array(sorted(best.values()))
    nm = L['nm'][rep]; ok = np.isfinite(nm)
    rep = rep[ok]; nm = nm[ok]; key = ik[rep]
    o = np.argsort(nm)
    return rep[o], key[o], nm[o]

# ===================================================================================
#  The two evidence channels
# ===================================================================================
def clean(mz, it):
    return _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                  CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)

def lib_sim(L, specs, target):
    """CLASS 1: direct match against library spectra of the same neutral mass."""
    cand = lib_window(L, target, target*CFG.PPM_WIN/1e6)
    if len(cand) == 0: return {}
    agg = {}
    for mz, it in specs:
        qm, qp = clean(mz, it)
        if len(qm) == 0: continue
        sc = search(qm, qp, cand, L['off'], L['mz'], L['it'],
                    CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT, 1)
        for c, s in zip(cand, sc):
            k = L['ik'][c]
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return agg

def analog_sim(L, specs, target, rep, rep_key, rep_nm):
    """CLASS 2: mass-SHIFTED match over a wide window. Relatives of the unknown fragment
       into the same ions offset by the mass difference, so they still match."""
    lo = np.searchsorted(rep_nm, target-CFG.ANALOG_WIN, 'left')
    hi = np.searchsorted(rep_nm, target+CFG.ANALOG_WIN, 'right')
    cand = rep[lo:hi]
    if len(cand) == 0: return []
    shift = (target - rep_nm[lo:hi]).astype(np.float32)
    ckey = rep_key[lo:hi]; agg = {}
    for mz, it in specs:
        qm, qp = clean(mz, it)
        if len(qm) == 0: continue
        sc = search_shift(qm, qp, cand, L['off'], L['mz'], L['it'],
                          CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                          CFG.INT_POWER, CFG.ENT_WEIGHT, 1, shift)
        for c, k, s in zip(cand, ckey, sc):
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return sorted(agg.items(), key=lambda x: -x[1])[:CFG.N_ANALOG]

## 6. CANDIDATE POOL

Сборка пула из 712k структур: COCONUT (готовые фингерпринты из `coco_fp.npy`) + train.parquet (считаем сами через RDKit). Фингерпринт = ECFP4(4096) ‖ ECFP6(4096) ‖ RDKitFP(2048) ‖ MACCS(167), обрезанный по `fp_bits.npy` до 6930 бит. Класс `Pool` хранит структуры, отсортированные по массе, с быстрым поиском в ppm-окне.

In [6]:
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Descriptors import ExactMolWt
from multiprocessing import Pool as MPool
RDLogger.DisableLog('rdApp.*')

BITS = np.load(find('fp_bits.npy'))
_g = {}
def _fp_init():
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)

def fp_and_mass(smi):
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        fp = np.concatenate([_g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[BITS]
        return fp, float(ExactMolWt(m))
    except Exception:
        return None

class Pool:
    """Candidates + fingerprints, sorted by exact mass."""
    def __init__(s, fp, mass, keys, smiles, nbits):
        o = np.argsort(mass)
        s._fp = fp[o]; s.mass = mass[o]
        s.keys = np.asarray(keys, dtype=object)[o]
        s.smiles = np.asarray(smiles, dtype=object)[o]
        s.nbits = nbits
        s.k2i = {k: i for i, k in enumerate(s.keys)}
    def window(s, t, ppm):
        a = np.searchsorted(s.mass, t*(1-ppm/1e6), 'left')
        b = np.searchsorted(s.mass, t*(1+ppm/1e6), 'right')
        return np.arange(a, b)
    def fps(s, idx):
        return np.unpackbits(np.asarray(s._fp[idx]), axis=1)[:, :s.nbits]

def build_pool():
    t0 = time.time()
    d = os.path.dirname(find('coco_fp.npy'))
    cm = pickle.load(open(d + '/coco_meta.pkl', 'rb'))
    co_fp = np.load(d + '/coco_fp.npy'); co_mass = np.load(d + '/coco_mass.npy')
    print(f'COCONUT half: {len(co_mass):,} structures', flush=True)

    tr = pq.read_table(TRAIN, columns=['inchikey14', 'normalized_smiles']).to_pandas()
    tr = tr.dropna().drop_duplicates('inchikey14')
    tr = tr[~tr.inchikey14.isin(set(cm['keys']))]
    print(f'training structures to fingerprint: {len(tr):,}  (~5 min)', flush=True)
    with MPool(4) as mp:
        res = mp.map(fp_and_mass, list(tr.normalized_smiles), chunksize=500)
    ok = [i for i, r in enumerate(res) if r is not None]
    tr_fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
    tr_mass = np.array([res[i][1] for i in ok])
    tr_keys = tr.inchikey14.values[ok]; tr_smi = tr.normalized_smiles.values[ok]

    fp = np.vstack([co_fp, tr_fp])
    mass = np.concatenate([co_mass, tr_mass])
    keys = np.concatenate([np.asarray(cm['keys'], dtype=object), tr_keys])
    smis = np.concatenate([np.asarray(cm['smiles'], dtype=object), tr_smi])
    good = np.isfinite(mass)
    print(f'pool: {int(good.sum()):,} structures   ({time.time()-t0:.0f}s)', flush=True)
    return Pool(fp[good], mass[good], keys[good], smis[good], cm['nbits'])

## 7. RANK FEATURES

`rank_features` собирает матрицу из 25 фич для каждой пары (молекула, кандидат): library similarity, analog Tanimoto (взвешенный по `sim^p`), статистики по аналогам, опциональные fingerprint-model logits и in-silico fragmentation score. Все фичи нормированы (rank-normalize, z-score), чтобы HistGradientBoosting мог их сравнивать.

In [7]:
import numpy as np

N_ANALOG = 80
P_SIM    = 3.0
N_FEAT   = 25

def _rank_norm(x):
    o=np.argsort(-x); r=np.empty(len(x)); r[o]=np.arange(len(x)); return r/max(1,len(x)-1)

def _z(x):
    s=x.std()
    return (x-x.mean())/s if s>1e-9 else np.zeros_like(x)

def rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):
    """cand_fp (nc,nbits), cand_lib (nc,) library similarity (0 if none),
       analog_fp (na,nbits), analog_sim (na,) descending,
       model_logits (nbits,) or None -> fingerprint-model evidence,
       frag (nc,) or None -> in-silico fragmentation explain-score (MetFrag-lite).
       Returns X (nc, N_FEAT)."""
    nc = cand_fp.shape[0]
    cf = cand_fp.astype(np.float32); cs = cf.sum(1)
    lv = np.asarray(cand_lib, np.float32)
    lvmax = float(lv.max()) if nc else 0.0
    if analog_fp is not None and len(analog_sim):
        af = analog_fp.astype(np.float32); asum = af.sum(1)
        inter = cf @ af.T
        tan = inter/(cs[:,None]+asum[None,:]-inter+1e-9)
        w = np.clip(np.asarray(analog_sim,np.float32),0,None)
        ap = (tan*(w**P_SIM)[None,:]).max(1)
        a1 = (tan*w[None,:]).max(1)
        best_tan = tan.max(1); top_tan = tan[:,0]; top_sim = float(w[0])
        mean_tan = (tan*(w**P_SIM)[None,:]).sum(1)/((w**P_SIM).sum()+1e-9)
    else:
        ap=a1=best_tan=top_tan=mean_tan=np.zeros(nc,np.float32); top_sim=0.0
    apmax = float(ap.max()) if nc else 0.0
    if model_logits is not None:
        raw = cf @ np.asarray(model_logits, np.float32)       # exact Bayes LL up to a constant
        nrm = raw/np.sqrt(np.maximum(cs,1.0))                 # length-corrected variant
        mfeat = [_z(raw), _rank_norm(raw), raw-raw.max(), _z(nrm), _rank_norm(nrm),
                 (raw==raw.max()).astype(np.float32)]
    else:
        mfeat = [np.zeros(nc,np.float32)]*6
    if frag is not None:
        fr = np.asarray(frag, np.float32)
        ffeat = [fr, _rank_norm(fr), fr-fr.max() if nc else fr, _z(fr)]
    else:
        ffeat = [np.zeros(nc,np.float32)]*4
    return np.column_stack([
        lv, _rank_norm(lv), np.full(nc,lvmax), lv-lvmax, (lv>0).astype(float),
        ap, _rank_norm(ap), np.full(nc,apmax), ap-apmax,
        a1, best_tan, top_tan, mean_tan, np.full(nc,top_sim),
        np.full(nc, np.log(max(nc,1))),
        *mfeat, *ffeat,
    ]).astype(np.float32)

## 8. METFRAG-LITE

In-silico фрагментация: `fragment_masses` ломает 1–2 связи в молекуле и собирает массы всех компонент (плюс ±2 H-перестройки). `explain_score` считает, какая доля (sqrt) интенсивности наблюдаемых пиков объясняется теоретическими фрагментными ионами. Независимый от analog propagation канал — их корреляция всего 0.058.

In [8]:
import numpy as np
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

AMU = {'C':12.0,'H':1.00782503207,'N':14.0030740048,'O':15.9949146196,'P':30.97376163,
       'S':31.97207100,'F':18.99840322,'Cl':34.96885268,'Br':78.9183371,'I':126.904473,
       'Na':22.9897692809,'K':38.96370668,'Si':27.9769265325,'B':11.0093054,'Se':79.9165213}
H = AMU['H']; PROTON = H - 0.00054857990

def mol_graph(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    n = m.GetNumAtoms()
    w = np.zeros(n)
    for a in m.GetAtoms():
        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs()*H
    if (w == 0).any(): return None
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]
    return w, bonds, n

def _components(n, bonds, drop):
    adj = [[] for _ in range(n)]
    for i,(a,b) in enumerate(bonds):
        if i in drop: continue
        adj[a].append(b); adj[b].append(a)
    seen = np.zeros(n, bool); comps=[]
    for s in range(n):
        if seen[s]: continue
        stack=[s]; seen[s]=True; cur=[s]
        while stack:
            u=stack.pop()
            for v in adj[u]:
                if not seen[v]: seen[v]=True; stack.append(v); cur.append(v)
        comps.append(cur)
    return comps

def fragment_masses(smi, max_breaks=2, max_bonds=34):
    """Neutral fragment masses from breaking 1 or 2 bonds."""
    g = mol_graph(smi)
    if g is None: return np.zeros(0)
    w, bonds, n = g
    nb = len(bonds)
    if nb == 0 or nb > max_bonds: return np.array([w.sum()])
    out = {w.sum()}
    for i in range(nb):
        for c in _components(n, bonds, {i}):
            out.add(float(w[c].sum()))
    if max_breaks >= 2:
        for i in range(nb):
            for j in range(i+1, nb):
                for c in _components(n, bonds, {i, j}):
                    out.add(float(w[c].sum()))
    return np.array(sorted(out))

def explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2,-1,0,1,2)):
    """Fraction of total (sqrt) intensity explained by some fragment ion."""
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    ion = []
    for dh in h_shifts:
        ion.append(frag_mass + dh*H + (PROTON if mode > 0 else -PROTON))
    ion = np.sort(np.concatenate(ion))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx+off, 0, len(ion)-1)
        ok |= np.abs(ion[k]-peak_mz) <= tol
    return float(w[ok].sum()/tot)

## 9. FRAG SCORES

Параллельный прогон MetFrag-lite по всем кандидатам молекулы: для каждого кандидата считаем `explain_score` по каждому спектру молекулы и берём max. Итоговый вектор `frag_scores` — независимый канал evidence для ранкера.

In [9]:
from multiprocessing import Pool as MPool

def _frag_masses(smi):
    try:  return fragment_masses(smi)
    except Exception: return np.zeros(0)

def frag_scores(cand_smiles, specs, mode, workers=4):
    """MetFrag-lite explain-score for every candidate, max over the molecule's spectra."""
    if not HAVE_RDKIT: return None
    with MPool(workers) as mp:
        frags = mp.map(_frag_masses, cand_smiles, chunksize=8)
    peaks = []
    for mz, it in specs:
        m2, i2 = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)
        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
    out = np.zeros(len(cand_smiles), np.float32)
    for j, f in enumerate(frags):
        out[j] = max((explain_score(f, a, b, mode=mode, tol=CFG.MZ_TOL) for a, b in peaks), default=0.0)
    return out

## 10. CALIBRATED RANKER

Загрузка готового датасета `rank_train.npz` (X, Y, M) и обучение HistGradientBoostingClassifier с весами: Class-1 симуляции получают вес `W1=0.42`, Class-2 — `1-W1=0.58`. W1 откалиброван по LB, а не по реальной доле классов.

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
z = np.load(find('rank_train.npz'))
NFEAT = z['X'].shape[1]
W = np.where(z['M'] == 0, CFG.W1, 1.0 - CFG.W1)      # M==0 -> Class-1 sim, M==1 -> Class-2 sim
RANKER = HistGradientBoostingClassifier(**CFG.GBM)
RANKER.fit(z['X'], z['Y'], sample_weight=W)
print(f'ranker fitted on {z["X"].shape[0]:,} candidate rows x {NFEAT} features   (W1={CFG.W1})')

ranker fitted on 142,762 candidate rows x 31 features   (W1=0.42)


## 11. INFERENCE & SUBMISSION

Основной прогон. Собирает пул кандидатов, грузит train-библиотеку, строит analog reference set. Для каждой тест-молекулы: объединяет все её спектры, считает lib_sim (Class 1) и analog_sim (Class 2), берёт кандидатов в ±10 ppm окне, считает frag_scores, собирает 25 фич и ранжирует через обученный RANKER. Пишет submission.csv с 25 SMILES на молекулу.

In [11]:
pool = build_pool()
print(f'candidate pool: {len(pool.mass):,} structures, {pool.nbits} fingerprint bits', flush=True)

L = load_library(TRAIN)
rep, rep_key, rep_nm = build_rep(L)
print(f'analog reference set: {len(rep):,} spectra (one per structure)', flush=True)

te = pq.read_table(TEST).to_pandas()
te['nm'] = neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id'))
print(f'{len(te):,} spectra / {len(mols):,} molecules to identify', flush=True)

rows, diag = [], []
for gi, (mid, sub) in enumerate(mols):
    nms  = sub.nm.values[np.isfinite(sub.nm.values)]
    smis = []
    if len(nms):
        target = float(np.median(nms))                       # fuse all spectra of the molecule
        specs  = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]

        lib_hits = lib_sim(L, specs, target)                 # Class-1 evidence
        analogs  = analog_sim(L, specs, target, rep, rep_key, rep_nm)   # Class-2 evidence

        cand = pool.window(target, CFG.PPM_WIN)
        if len(cand) == 0:
            cand = pool.window(target, CFG.PPM_FALLBACK)
        if len(cand):
            cfp = pool.fps(cand)
            lv  = np.array([lib_hits.get(pool.keys[c], 0.0) for c in cand], np.float32)
            ids, sims = [], []
            for k, s in analogs:
                i = pool.k2i.get(k, -1)
                if i >= 0: ids.append(i); sims.append(s)
            afp = pool.fps(np.array(ids)) if ids else None
            fsc = frag_scores([pool.smiles[c] for c in cand], specs,
                              float(np.mean([1.0 if m == 'positive' else -1.0
                                             for m in sub.ionization_mode])))
            X   = rank_features(cfp, lv, afp, np.array(sims, np.float32), None, fsc)[:, :NFEAT]
            p   = RANKER.predict_proba(X)[:, 1]
            order = np.argsort(-p)[:CFG.TOPN]
            smis  = [pool.smiles[cand[i]] for i in order]
            diag.append((mid, target, len(cand), float(lv.max()),
                         float(sims[0]) if sims else 0.0, float(p[order[0]])))
    if not smis: smis = ['CCO']
    rows.append((mid, ';'.join(smis[:CFG.TOPN])))
    if gi % 50 == 0: print(f'  {gi}/{len(mols)}  {time.time()-T0:.0f}s', flush=True)

submission = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
samp = pd.read_csv(SAMPLE)
submission = samp[['molecule_id']].merge(submission, on='molecule_id', how='left')
submission['smiles'] = submission['smiles'].fillna('CCO')

assert len(submission) == len(samp)
assert submission.molecule_id.duplicated().sum() == 0
assert submission.smiles.isnull().sum() == 0
assert submission.smiles.str.split(';').map(len).max() <= 25
submission.to_csv('submission.csv', index=False)
print(f'\nwrote submission.csv  {submission.shape}   total {time.time()-T0:.0f}s')
submission.head()

COCONUT half: 436,389 structures
training structures to fingerprint: 275,810  (~5 min)
pool: 712,199 structures   (263s)
candidate pool: 712,199 structures, 6930 fingerprint bits
library: 2,539,608 spectra / 275,810 structures  (28s)
analog reference set: 275,632 spectra (one per structure)
1,213 spectra / 400 molecules to identify


ValueError: X has 25 features, but HistGradientBoostingClassifier is expecting 31 features as input.

## 12. DIAGNOSTICS

Описательная статистика по молекулам (число кандидатов, лучший library sim, лучший analog sim, top probability) и 4 графика: распределение кандидатов, library similarity, analog similarity, scatter двух каналов evidence.

In [ ]:
import matplotlib.pyplot as plt
d = pd.DataFrame(diag, columns=['molecule_id','neutral_mass','n_candidates',
                                'best_library_sim','best_analog_sim','top_prob'])
print(d[['n_candidates','best_library_sim','best_analog_sim','top_prob']].describe().round(3).to_string())

fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ax[0].hist(d.n_candidates, bins=40, color='#4C72B0'); ax[0].set_title('candidates per molecule'); ax[0].set_xlabel('n')
ax[1].hist(d.best_library_sim, bins=40, color='#DD8452'); ax[1].set_title('best library similarity'); ax[1].set_xlabel('entropy sim')
ax[2].hist(d.best_analog_sim, bins=40, color='#55A868'); ax[2].set_title('best analog similarity'); ax[2].set_xlabel('entropy sim (mass-shifted)')
ax[3].scatter(d.best_library_sim, d.best_analog_sim, s=8, alpha=.5, color='#C44E52')
ax[3].set_xlabel('library sim'); ax[3].set_ylabel('analog sim'); ax[3].set_title('the two evidence channels')
for a in ax: a.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

# molecules where the library is confident are likely Class 1; the rest lean on analogs
likely_c1 = (d.best_library_sim > 0.85).mean()
print(f'\nmolecules with a confident library hit (sim > 0.85): {likely_c1:.1%}'
      f'  -> the other {1-likely_c1:.1%} are carried by analog propagation')